# Gold Layer — Compliance Engine
**Layer:** Gold (aggregated, business-ready metrics)
**Source:** silver.silver_metadata
**Purpose:** Structural compliance KPIs — identifies tables deviating from the 10-column standard.

In [ ]:
# Catalog widget — injected by Job base_parameters or set manually
dbutils.widgets.text("catalog", "metadata_governance", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

## Step 1 — Read Silver Metadata

In [ ]:
silver_df = spark.table(f"{catalog}.silver.silver_metadata")
silver_count = silver_df.count()
print(f"Silver rows loaded: {silver_count}")
assert silver_count > 0, "Silver table is empty — cannot build Gold layer"

## Step 2 — Compute Table-Level Compliance Metrics

Count columns per table and compare against the structural standard of 10 columns.

In [ ]:
from pyspark.sql.functions import col, countDistinct, when, lit, current_timestamp, current_date

STANDARD_COLUMN_COUNT = 10

table_metrics = silver_df.groupBy(
    "database_name", "schema_name", "table_name"
).agg(
    countDistinct("column_name").alias("column_count")
).withColumn(
    "standard_column_count", lit(STANDARD_COLUMN_COUNT)
).withColumn(
    "is_compliant",
    when(col("column_count") == STANDARD_COLUMN_COUNT, True).otherwise(False)
).withColumn(
    "compliance_status",
    when(col("column_count") == STANDARD_COLUMN_COUNT, lit("COMPLIANT")).otherwise(lit("NON_COMPLIANT"))
).withColumn(
    "deviation", col("column_count") - lit(STANDARD_COLUMN_COUNT)
).withColumn(
    "report_timestamp", current_timestamp()
).withColumn(
    "report_date", current_date()
)

display(table_metrics.orderBy("is_compliant", "table_name"))
print(f"Total tables analysed: {table_metrics.count()}")

## Step 3 — Compute Summary KPIs

Aggregate compliance statistics for the shareholder dashboard.

In [ ]:
from pyspark.sql import Row

total_tables = table_metrics.count()
compliant_tables = table_metrics.filter(col("is_compliant") == True).count()
non_compliant_tables = total_tables - compliant_tables
compliance_rate = round((compliant_tables / total_tables) * 100, 2) if total_tables > 0 else 0

print(f"Total tables:         {total_tables}")
print(f"Compliant tables:     {compliant_tables}")
print(f"Non-compliant tables: {non_compliant_tables}")
print(f"Compliance rate:      {compliance_rate}%")

kpi_row = Row(
    total_tables=total_tables,
    compliant_tables=compliant_tables,
    non_compliant_tables=non_compliant_tables,
    compliance_rate_pct=compliance_rate,
    standard_column_count=STANDARD_COLUMN_COUNT,
    total_silver_rows=silver_count
)
kpi_df = spark.createDataFrame([kpi_row]) \
    .withColumn("report_timestamp", current_timestamp()) \
    .withColumn("report_date", current_date())

display(kpi_df)

## Step 4 — Save Gold Tables

In [ ]:
table_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.gold_table_compliance")

kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.gold_compliance_kpis")

print(f"Gold tables written successfully to {catalog}.gold")

## Step 5 — Compliance Alert for Non-Compliant Tables

In [ ]:
non_compliant_df = table_metrics.filter(col("is_compliant") == False)

if non_compliant_df.count() > 0:
    print(f"[COMPLIANCE ALERT] {non_compliant_df.count()} tables do not meet the structural standard of {STANDARD_COLUMN_COUNT} columns.")
    display(non_compliant_df.select(
        "database_name", "schema_name", "table_name",
        "column_count", "standard_column_count", "deviation", "compliance_status"
    ).orderBy("deviation"))
else:
    print("All tables comply with the structural standard.")

print("Gold Compliance Engine completed successfully.")

## Step 6 — Validate Gold Output

In [ ]:
gold_compliance = spark.table(f"{catalog}.gold.gold_table_compliance")
gold_kpis = spark.table(f"{catalog}.gold.gold_compliance_kpis")
print(f"gold_table_compliance rows: {gold_compliance.count()}")
print(f"gold_compliance_kpis rows:  {gold_kpis.count()}")
display(gold_kpis)